In [4]:
import re
import openai
from langsmith import traceable, get_current_run_tree
from qdrant_client import QdrantClient

from dotenv import load_dotenv

load_dotenv("../.env")

COLLECTION_NAME = "Recipes-collection-01"

qdrant_client = QdrantClient(url="http://localhost:6333")

@traceable(
  name="embed_query",
  run_type="embedding",
  metadata={
    "ls_provider": "openai",
    "ls_model_name": "text-embedding-3-small"
  }
)
def get_embedding(text, model="text-embedding-3-small"):
  response = openai.embeddings.create(
    input=text,
    model=model
  )
  current_run = get_current_run_tree()
  if current_run:
    current_run.metadata["usage_metadata"] = {
      "input_tokens": response.usage.prompt_tokens,
      "total_tokens": response.usage.total_tokens,
    }
  return response.data[0].embedding

@traceable(
  name="retrieve_data",
  run_type="retriever"
)
def retrieve_data(query, k=5):
  query_embedding = get_embedding(query)
  results = qdrant_client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_embedding,
    limit=k,
  )
  retrieved = []
  for result in results.points:
    payload = result.payload
    retrieved.append({
      "id": int(payload["RecipeId"]),
      "name": payload["Name"],
      "score": result.score,
      "rating": payload["bayesian_rating"],
      "n_ratings": payload["n_ratings"],
      "calories": payload.get("Calories"),
      "protein": payload.get("ProteinContent"),
      "carbs": payload.get("CarbohydrateContent"),
      "fat": payload.get("FatContent"),
      "total_time": payload.get("TotalTime"),
      "ingredients": payload.get("RecipeIngredientParts") or [],
      "instructions": payload.get("RecipeInstructions") or [],
    })
  return retrieved

@traceable(
  name="format_time",
  run_type="parser"
)
def format_time(iso_duration):
  # "PT1H30M" -> "90 min". Returns None if there's nothing usable.
  if not iso_duration:
    return None
  hours = re.search(r"(\d+)H", iso_duration)
  minutes = re.search(r"(\d+)M", iso_duration)
  total = (int(hours.group(1)) * 60 if hours else 0) + (int(minutes.group(1)) if minutes else 0)
  return f"{total} min" if total else None

@traceable(
  name="format_retrieved_context",
  run_type="prompt"
)
def process_context(retrieved, max_steps_chars=300):
  blocks = []
  for i, r in enumerate(retrieved, start=1):
    nutrition = []
    if r["calories"] is not None:
      nutrition.append(f"{round(r['calories'])} kcal")
    if r["protein"] is not None:
      nutrition.append(f"P {round(r['protein'])}g")
    if r["carbs"] is not None:
      nutrition.append(f"C {round(r['carbs'])}g")
    if r["fat"] is not None:
      nutrition.append(f"F {round(r['fat'])}g")

    ingredients = ", ".join(str(x) for x in r["ingredients"] if x)

    steps = " ".join(str(s) for s in r["instructions"] if s)
    if len(steps) > max_steps_chars:
      steps = steps[:max_steps_chars].rstrip() + "..."

    blocks.append(
      f"[{i}] {r['name']} (id: {r['id']})\n"
      f"  rating: {r['rating']:.1f} ({r['n_ratings']} reviews)\n"
      f"  nutrition: {' | '.join(nutrition) or 'n/a'}\n"
      f"  total time: {format_time(r['total_time']) or 'n/a'}\n"
      f"  ingredients: {ingredients}\n"
      f"  steps: {steps}"
    )

  return "\n\n".join(blocks)

@traceable(
  name="build_prompt",
  run_type="prompt"
)
def build_system_prompt(context):
  return f"""
You are a helpful cooking assistant. You help people decide what to cook by recommending recipes from the ones available below.

Instructions:
- Only recommend recipes from the available recipes. Never invent recipes, ingredients, or nutrition values.
- Refer to recipes by their name; you may add the id in parentheses so it can be looked up.
- If the question has constraints (calories, time, an ingredient to include or avoid, a meal type), respect them and prefer recipes that match.
- If none of the available recipes fit the request well, say so honestly instead of forcing a poor match.
- The steps shown are only a short preview, not the full method, so don't present them as complete instructions.
- Keep the answer concise and friendly. Do not use markdown.

Available recipes:
{context}
"""

@traceable(
  name="generate_answer",
  run_type="llm",
  metadata={
    "ls_provider": "openai",
    "ls_model_name": "gpt-5.4-nano"
  }
)
def generate_answer(system_prompt, question):
  response = openai.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": question},
    ],
    reasoning_effort="none",
  )
  current_run = get_current_run_tree()
  if current_run:
    current_run.metadata["usage_metadata"] = {
      "input_tokens": response.usage.prompt_tokens,
      "output_tokens": response.usage.completion_tokens,
      "total_tokens": response.usage.total_tokens
    }
  return response.choices[0].message.content

@traceable(
  name="rag_pipeline",
  run_type="chain"
)
def rag_pipeline(question, k=5):
  context = process_context(retrieve_data(question, k))
  system_prompt = build_system_prompt(context)
  return generate_answer(system_prompt, question)

In [6]:
print(rag_pipeline("Find recipes with eggplants"))

Here are the available recipes that use eggplant:

1) Eggplant Lasagna (247267)  
2) Pickled Eggplant (Aubergine) (12107)  
3) Imam Baildi Aka Stuffed Eggplant (Aubergine) (83756)  
4) Eggplant Roll Stuffed with Ricotta and Vegetables (54495)  
5) Eggplant (Aubergine) With Raw Garlic (42280)

Want a quick option (under ~30 min), something vegetarian, or something specific like pickling or baked?
